# ESM3 蛋白质结构预测工作流（DALI格式输出）

这个工作流程将：
1. 读取蛋白质序列文件（FASTA 格式，来自 Prokka 或其他工具）
2. 使用 ESM3 进行结构预测（支持所有官方参数）
3. 生成符合 DALI 输入标准的 PDB 文件

## 系统要求
- JupyterLab/JupyterHub 服务器环境（推荐使用 GPU）
- 约 10-20 GB 磁盘空间
- 运行时间取决于序列数量和长度

## 输入要求
- **蛋白质序列文件**：FASTA 格式（`.faa`, `.fa`, `.fasta`）
- 可以来自 Prokka 输出或其他基因注释工具

## 更新说明
- ✅ 使用共享代码，消除重复
- ✅ 支持所有ESM3官方参数
- ✅ 自动依赖管理

## 1. 环境设置与依赖安装

In [ ]:
# 使用共享工具初始化环境（自动安装依赖、设置路径）
import sys
from pathlib import Path

# 检测运行环境
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    WORK_DIR = Path('/content/esm3_workflow')
else:
    WORK_DIR = None  # 将由setup_esm3_notebook设置

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入并设置环境
from protflow.utils.notebook_utils import setup_esm3_notebook
from protflow.prediction.esm3_predict import load_esm3_small, predict_structures_from_fasta, ESM3GenerationConfig

# 设置环境（自动检查和安装依赖）
if not IN_COLAB:
    paths = setup_esm3_notebook(work_dir_name='esm3_runs')
    PROJECT_ROOT = paths['PROJECT_ROOT']
    WORK_DIR = paths['WORK_DIR']
    DATA_DIR = paths['DATA_DIR']
else:
    WORK_DIR.mkdir(exist_ok=True, parents=True)
    PROJECT_ROOT = WORK_DIR
    DATA_DIR = WORK_DIR

print(f"\n✓ 环境初始化完成")
print(f"  工作目录: {WORK_DIR}")

## 2. 选择输入文件

上传或指定你的蛋白质序列文件（FASTA 格式）

In [ ]:
from Bio import SeqIO

if IN_COLAB:
    print('请上传蛋白质序列文件...')
    uploaded = files.upload()
    input_faa = Path(list(uploaded.keys())[0])
else:
    # 本地环境：指定文件路径
    input_faa = DATA_DIR / 'inputs' / 'proteins.faa'  # 请替换为您的文件路径
    if not input_faa.exists():
        print(f"⚠️ 请设置正确的输入文件路径: {input_faa}")
    else:
        print(f"✓ 使用文件: {input_faa}")

if input_faa.exists():
    sequences = list(SeqIO.parse(input_faa, 'fasta'))
    print(f"✓ 读取到 {len(sequences)} 条序列")
else:
    sequences = []
    print("⚠️ 未找到输入文件")

## 3. 配置ESM3参数

使用`ESM3GenerationConfig`配置所有官方参数：

In [ ]:
# 配置参数（支持所有ESM3官方参数）
gen_config = ESM3GenerationConfig(
    track='structure',      # 'sequence', 'structure', 'function'
    num_steps=8,           # 生成步数（8-16，越大质量可能越好但更慢）
    temperature=None        # 温度参数（可选，None使用模型默认值）
)

# 序列长度过滤
MIN_SEQ_LENGTH = 30
MAX_SEQ_LENGTH = 400

# 创建输出目录
pdb_dir = WORK_DIR / "esm3_structures"
dali_dir = WORK_DIR / "dali_ready"
pdb_dir.mkdir(exist_ok=True, parents=True)
dali_dir.mkdir(exist_ok=True, parents=True)

print(f"ESM3配置:")
print(f"  track: {gen_config.track}")
print(f"  num_steps: {gen_config.num_steps}")
print(f"  temperature: {gen_config.temperature}")
print(f"\n序列长度范围: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH} aa")
print(f"  PDB 输出: {pdb_dir}")
print(f"  DALI 输出: {dali_dir}")

## 4. 运行结构预测

⚠️ 这一步可能需要较长时间，取决于序列数量

In [ ]:
# 使用共享模块进行结构预测
if input_faa.exists() and len(sequences) > 0:
    print(f"\n开始预测 {len(sequences)} 个蛋白质的结构...")
    
    results = predict_structures_from_fasta(
        fasta_file=input_faa,
        out_dir=pdb_dir,
        generation_config=gen_config,
        min_seq_length=MIN_SEQ_LENGTH,
        max_seq_length=MAX_SEQ_LENGTH,
        show_progress=True,
        skip_existing=True
    )
    
    print(f"\n✅ 预测完成!")
    print(f"  成功: {results['success']}")
    print(f"  跳过: {results['skipped']}")
    print(f"  错误: {results['errors']}")
    print(f"  过滤: {results['filtered']}")
    
    # 获取所有生成的PDB文件
    pdb_files = list(pdb_dir.glob('*.pdb'))
    print(f"\n✓ 共生成 {len(pdb_files)} 个PDB文件")
else:
    print("⚠️ 无有效输入文件或序列，跳过预测")
    pdb_files = []

## 5. 准备 DALI 文件

将 PDB 文件转换为 DALI 兼容格式

In [ ]:
# 使用后端模块准备DALI文件（所有业务逻辑在后端）
from protflow.prediction.dali import prepare_pdb_for_dali
from pathlib import Path

if len(pdb_files) > 0:
    print("准备 DALI 文件...\n")
    
    # 使用后端模块准备DALI文件
    dali_output_dir = prepare_pdb_for_dali(
        pdb_files=[Path(f) for f in pdb_files],
        output_dir=dali_dir,
        generate_mapping=True
    )
    
    print(f"\n✅ DALI 文件准备完成!")
    print(f"  位置: {dali_output_dir}")
    print(f"  文件数: {len(pdb_files)}")
    print(f"  映射表: {dali_output_dir / 'pdb_id_mapping.tsv'}")
else:
    print("⚠️ 无PDB文件，跳过DALI文件准备")

## 6. 查看结果

In [ ]:
print(f"\n{'='*60}")
print("结果摘要")
print(f"{'='*60}")
print(f"\n工作目录: {WORK_DIR}")
print(f"\n1. ESM3 结构 ({len(list(pdb_dir.glob('*.pdb')))} 个 PDB 文件)")
print(f"   {pdb_dir}")
print(f"\n2. DALI 文件 ({len(list(dali_dir.glob('*.ent')))} 个 ENT 文件)")
print(f"   {dali_dir}")

total_size = sum(f.stat().st_size for f in pdb_dir.glob('*.pdb'))
total_size += sum(f.stat().st_size for f in dali_dir.glob('*.ent'))
print(f"\n总磁盘使用: {total_size / 1024 / 1024:.1f} MB")

## 7. 下载结果（Colab 用户）

In [ ]:
if IN_COLAB:
    import zipfile
    
    print("打包结果...")
    zip_path = WORK_DIR / 'esm3_results.zip'
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for pdb_file in pdb_dir.glob('*.pdb'):
            zipf.write(pdb_file, arcname=f"structures/{pdb_file.name}")
        for dali_file in dali_dir.glob('*'):
            if dali_file.is_file():
                zipf.write(dali_file, arcname=f"dali/{dali_file.name}")
    
    files.download(str(zip_path))
    print(f"✓ 已下载: {zip_path.name}")
else:
    print("结果已保存在服务器:")
    print(f"  {WORK_DIR.resolve()}")